# Kaggle Content Ingestion — GeeksforGeeks + W3Schools

Downloads these Kaggle datasets, keeps only **AI / Data / Cloud** content, standardizes the schema, cleans duplicates, and saves JSON to `data/raw/`.

Datasets:
- `naidukarthi2193/geeks-for-geeks-articles-dataset`
- `haohoangofficial/w3school-crawling-2025`

The notebook inspects the actual downloaded files/columns instead of assuming a fixed Kaggle schema.


## 1. Install packages

In [1]:
# %pip install -q kagglehub pandas pyarrow

## 2. Imports

In [2]:
from pathlib import Path
import json
import re
import pandas as pd
import kagglehub

d:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Output paths
This notebook assumes it is saved under `notebooks/geeks_for_geeks_w3schools/`.

In [3]:
DATA_DIR = Path("../../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

GFG_OUTPUT = DATA_DIR / "geeksforgeeks_articles.json"
W3_OUTPUT = DATA_DIR / "w3schools_content.json"

print("Raw data folder:", DATA_DIR.resolve())

Raw data folder: D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\data\raw


## 4. Download both Kaggle datasets

In [4]:
GFG_DATASET = "naidukarthi2193/geeks-for-geeks-articles-dataset"
W3_DATASET = "haohoangofficial/w3school-crawling-2025"

gfg_path = Path(kagglehub.dataset_download(GFG_DATASET))
w3_path = Path(kagglehub.dataset_download(W3_DATASET))

print("GeeksforGeeks:", gfg_path)
print("W3Schools:", w3_path)

100%|██████████| 10.2M/10.2M [00:01<00:00, 6.84MB/s]

Extracting files...


100%|██████████| 767M/767M [02:38<00:00, 5.07MB/s] 

Extracting files...


GeeksforGeeks: C:\Users\Sarah\.cache\kagglehub\datasets\naidukarthi2193\geeks-for-geeks-articles-dataset\versions\1
W3Schools: C:\Users\Sarah\.cache\kagglehub\datasets\haohoangofficial\w3school-crawling-2025\versions\3


## 5. Inspect downloaded files

In [5]:
def list_files(folder):
    files = [p for p in folder.rglob("*") if p.is_file()]
    for p in files:
        print(p.relative_to(folder))
    return files

print("=== GeeksforGeeks ===")
gfg_files = list_files(gfg_path)

print("\n=== W3Schools ===")
w3_files = list_files(w3_path)

=== GeeksforGeeks ===
geekstest1.csv

=== W3Schools ===
w3school-crawling-2025\metadata.json
w3school-crawling-2025\html\0000a7c0-4e50-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\0004d491-4e50-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\00059e60-4cf2-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\0008f516-4cf2-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\000906a9-4e50-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\000d1fc2-4e50-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\000e40c5-4cf2-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\0011366d-4e50-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\0012064b-4cf2-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\001513bd-4e50-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\00159068-4cf2-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\0016ec42-4d21-11f0-9280-24b6fdf6f54e.html
w3school-crawling-2025\html\0019080c-4e50-11f0-9280-24b6fdf6f54e.ht

## 6. Load CSV / JSON / JSONL / Parquet files

In [7]:
def load_tabular_files(folder):
    frames = []

    for file in folder.rglob("*"):
        if not file.is_file():
            continue

        try:
            suffix = file.suffix.lower()

            # CSV
            if suffix == ".csv":
                try:
                    df = pd.read_csv(
                        file,
                        encoding="utf-8",
                        low_memory=False
                    )
                except UnicodeDecodeError:
                    print(f"UTF-8 failed for {file.name}, trying latin-1...")

                    df = pd.read_csv(
                        file,
                        encoding="latin-1",
                        low_memory=False
                    )

            # JSON Lines
            elif suffix == ".jsonl":
                df = pd.read_json(
                    file,
                    lines=True
                )

            # JSON
            elif suffix == ".json":
                try:
                    df = pd.read_json(file)
                except ValueError:
                    df = pd.read_json(
                        file,
                        lines=True
                    )

            # Parquet
            elif suffix == ".parquet":
                df = pd.read_parquet(file)

            # Ignore unsupported files such as HTML
            else:
                continue

            df["_source_file"] = file.name
            frames.append(df)

            print(f"Loaded {file.name}: {df.shape}")
            print("Columns:", df.columns.tolist())
            print()

        except Exception as e:
            print(f"Could not load {file.name}: {e}")

    if not frames:
        raise ValueError(
            f"No supported CSV/JSON/JSONL/Parquet files found in {folder}"
        )

    return pd.concat(
        frames,
        ignore_index=True,
        sort=False
    )


gfg_raw = load_tabular_files(gfg_path)

print("\nGeeksforGeeks total shape:", gfg_raw.shape)

UTF-8 failed for geekstest1.csv, trying latin-1...
Loaded geekstest1.csv: (24489, 6)
Columns: ['url', 'title', 'rating', 'content', 'tags', '_source_file']


GeeksforGeeks total shape: (24489, 6)


## 7. Preview raw data

In [ ]:
display(gfg_raw.head())
display(w3_raw.head())

## 8. AI / Data / Cloud keywords

In [ ]:
TOPIC_KEYWORDS = {
    "AI": [
        "artificial intelligence", "machine learning", "deep learning",
        "generative ai", "genai", "large language model", "large language models",
        "llm", "llms", "natural language processing", "nlp",
        "computer vision", "neural network", "neural networks",
        "transformer", "transformers", "tensorflow", "pytorch", "scikit-learn"
    ],
    "Data": [
        "data science", "data scientist", "data engineering", "data engineer",
        "data analytics", "data analysis", "big data", "data pipeline",
        "data pipelines", "etl", "elt", "data warehouse", "data warehousing",
        "data lake", "data lakes", "database", "databases", "sql",
        "postgresql", "mysql", "mongodb", "spark", "apache spark",
        "airflow", "apache airflow", "dbt", "pandas"
    ],
    "Cloud": [
        "cloud computing", "cloud architecture", "cloud infrastructure",
        "cloud engineering", "cloud engineer", "aws", "amazon web services",
        "azure", "microsoft azure", "google cloud", "google cloud platform",
        "gcp", "serverless", "kubernetes", "docker", "cloud native"
    ]
}

## 9. Discover equivalent columns across the two datasets

In [ ]:
COLUMN_ALIASES = {
    "title": ["title", "article_title", "article title", "name", "heading", "topic"],
    "author": ["author", "authors", "writer", "author_name"],
    "publication_date": ["publication_date", "published_date", "publish_date",
                         "date", "last_updated", "last update", "updated", "timestamp"],
    "url": ["url", "link", "article_url", "article link", "href"],
    "content": ["content", "text", "article", "article_content", "body",
                "description", "tutorial_content", "paragraph"],
    "category": ["category", "categories", "topic_category", "section", "subject"],
    "tags": ["tags", "tag", "keywords"]
}

def find_column(df, aliases):
    normalized = {
        str(col).strip().lower().replace("-", "_"): col
        for col in df.columns
    }
    for alias in aliases:
        key = alias.strip().lower().replace("-", "_")
        if key in normalized:
            return normalized[key]
    return None

def discover_mapping(df):
    return {field: find_column(df, aliases)
            for field, aliases in COLUMN_ALIASES.items()}

gfg_mapping = discover_mapping(gfg_raw)
w3_mapping = discover_mapping(w3_raw)

print("GFG mapping:", gfg_mapping)
print("W3 mapping:", w3_mapping)

## 10. Topic classifier

In [ ]:
def clean_text(value):
    return "" if pd.isna(value) else str(value).strip()

def classify_topics(text):
    text = clean_text(text).lower()
    matched = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        for keyword in keywords:
            if len(keyword) <= 4 and keyword.replace("-", "").isalnum():
                if re.search(rf"(?<!\\w){re.escape(keyword)}(?!\\w)", text):
                    matched.append(topic)
                    break
            elif keyword in text:
                matched.append(topic)
                break
    return matched

def build_search_text(row, mapping):
    pieces = []
    for field in ["title", "category", "tags", "content"]:
        col = mapping.get(field)
        if col and col in row.index:
            value = clean_text(row[col])
            if field == "content":
                value = value[:5000]
            pieces.append(value)
    return " ".join(pieces)

## 11. Keep only AI / Data / Cloud records

In [ ]:
def filter_relevant(df, mapping, source_name):
    result = df.copy()
    result["_topics"] = result.apply(
        lambda row: classify_topics(build_search_text(row, mapping)), axis=1
    )
    result = result[result["_topics"].map(len) > 0].copy()
    print(f"{source_name}: {len(df):,} raw -> {len(result):,} relevant")
    return result

gfg_filtered = filter_relevant(gfg_raw, gfg_mapping, "GeeksforGeeks")
w3_filtered = filter_relevant(w3_raw, w3_mapping, "W3Schools")

## 12. Standardize publication dates to `YYYY-MM-DD`
If a dataset has no date, leave it blank.

In [ ]:
def normalize_dates(series):
    parsed = pd.to_datetime(series, errors="coerce", utc=True)
    return parsed.dt.strftime("%Y-%m-%d").fillna("")

## 13. Standardize to the project schema

In [ ]:
def get_series(df, col):
    if col and col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index, dtype="object")

def standardize(df, mapping, source_name):
    date_col = mapping.get("publication_date")
    dates = (normalize_dates(df[date_col]) if date_col
             else pd.Series([""] * len(df), index=df.index))

    return pd.DataFrame({
        "source": [source_name] * len(df),
        "category": df["_topics"].apply(lambda x: ", ".join(x)),
        "title": get_series(df, mapping.get("title")),
        "author": get_series(df, mapping.get("author")),
        "publication_date": dates,
        "description": [""] * len(df),
        "url": get_series(df, mapping.get("url")),
        "content": get_series(df, mapping.get("content")),
        "tags": get_series(df, mapping.get("tags"))
    }).reset_index(drop=True)

gfg_standardized = standardize(gfg_filtered, gfg_mapping, "GeeksforGeeks")
w3_standardized = standardize(w3_filtered, w3_mapping, "W3Schools")

display(gfg_standardized.head())
display(w3_standardized.head())

## 14. Basic data-quality cleaning

In [ ]:
def clean_standardized(df):
    result = df.copy()

    for col in result.columns:
        if result[col].dtype == "object":
            result[col] = result[col].apply(
                lambda x: x.strip() if isinstance(x, str) else x
            )

    result = result[(result["title"] != "") | (result["content"] != "")]

    with_url = result[result["url"] != ""].drop_duplicates(subset=["url"])
    without_url = result[result["url"] == ""].drop_duplicates(
        subset=["source", "title", "content"]
    )

    return pd.concat([with_url, without_url], ignore_index=True)

gfg_final = clean_standardized(gfg_standardized)
w3_final = clean_standardized(w3_standardized)

print("GFG final:", gfg_final.shape)
print("W3 final:", w3_final.shape)

## 15. Topic distributions

In [ ]:
print("=== GeeksforGeeks ===")
print(gfg_final["category"].str.split(", ").explode().value_counts())

print("\n=== W3Schools ===")
print(w3_final["category"].str.split(", ").explode().value_counts())

## 16. Manually validate random samples

In [ ]:
def show_sample(df, name, n=10):
    print(f"\n===== {name} =====")
    if df.empty:
        print("No matching records.")
        return

    for _, row in df.sample(min(n, len(df)), random_state=42).iterrows():
        print("-" * 80)
        print("TITLE:", row["title"])
        print("CATEGORY:", row["category"])
        print("URL:", row["url"])

show_sample(gfg_final, "GeeksforGeeks")
show_sample(w3_final, "W3Schools")

## 17. Save JSON files to `data/raw/`
If these are large, add them to `.gitignore` rather than committing generated data.

In [ ]:
def save_json(df, path):
    records = df.to_dict(orient="records")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(records):,} -> {path.resolve()}")

save_json(gfg_final, GFG_OUTPUT)
save_json(w3_final, W3_OUTPUT)

## 18. Verify the saved files

In [ ]:
for path in [GFG_OUTPUT, W3_OUTPUT]:
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print(path.name, ":", len(records), "records")
    if records:
        print(json.dumps(records[0], indent=2, ensure_ascii=False)[:2000])
    print("=" * 80)

## Next pipeline step

Keep these outputs in `data/raw/`. Your later profiling, cleaning, schema-validation, and join/transform stages can combine them with Medium, Pluralsight, and your teammates' sources.

For Azure, the raw outputs can later be placed in Blob Storage / ADLS Gen2 and orchestrated with Azure Data Factory.
